In [14]:
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split


from sklearn.preprocessing import StandardScaler

from feature_engine.outliers import Winsorizer

df = pd.read_csv('../data/raw/Customer-Churn-Records.csv')

In [15]:
X = df.drop(columns=['Exited'])
y = df['Exited']

In [16]:
numeric_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary', 'Point Earned']
winsor = Winsorizer(capping_method='iqr', tail='both', fold=1.5, variables=numeric_cols)
X_winsorized = winsor.fit_transform(X)

In [17]:

X_winsorized['Age'] = np.log1p(X_winsorized['Age'])

X_winsorized['Balance'] = np.log1p(X_winsorized['Balance'])

In [18]:
X_encoded = pd.get_dummies(X_winsorized, columns=['Geography', 'Gender', 'Card Type'], drop_first=True)
X_encoded['Satisfaction Score'] = X_encoded['Satisfaction Score'].astype('category').cat.codes

In [19]:
X_train_temp, X_test, y_train_temp, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

In [20]:
print(f"X_train_temp shape: {X_train_temp.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")

X_train_temp shape: (8000, 20)
X_val shape: (1000, 20)
X_test shape: (1000, 20)


In [21]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_temp[numeric_cols])
X_val_scaled = scaler.transform(X_val[numeric_cols])
X_test_scaled = scaler.transform(X_test[numeric_cols])

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=numeric_cols, index=X_train_temp.index)
X_val_scaled_df = pd.DataFrame(X_val_scaled, columns=numeric_cols, index=X_val.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=numeric_cols, index=X_test.index)

X_train_final = pd.concat([X_train_scaled_df, X_train_temp.drop(columns=numeric_cols)], axis=1)
X_val_final = pd.concat([X_val_scaled_df, X_val.drop(columns=numeric_cols)], axis=1)
X_test_final = pd.concat([X_test_scaled_df, X_test.drop(columns=numeric_cols)], axis=1)

print(f'Train shape: {X_train_final.shape}, Val shape: {X_val_final.shape}, Test shape: {X_test_final.shape}')

Train shape: (8000, 20), Val shape: (1000, 20), Test shape: (1000, 20)


In [22]:
df.to_csv('cleaned_data.csv', index=False)